# Floating-Point Numbers and Precision

This notebook was generated from the FreeCampus Python lesson source. Run cells from top to bottom, write predictions before execution, and change one thing at a time.

Source lesson: `courses/python-foundations/units/core-values-types/floating-point-precision.qmd`

- **Level:** Python Foundations · Unit 2
- **Estimated time:** 3–4 hours
- **You will learn:** Model measured quantities with floats, explain binary approximation, compare with tolerances, and format results without mistaking display rounding for exactness.
- **Practice in:** Google Colab, JupyterLab, or a local editor

## 1. Floats model quantities between whole numbers

A trail can be `4.75` kilometers long, a sensor can report `21.6` degrees, and an
experiment can take `0.003` seconds. Python normally represents these measured or
fractional values with `float`:

In [ ]:
trail_km = 4.75
temperature_c = 21.6
elapsed_seconds = 0.003

print(type(trail_km).__name__)
print(type(temperature_c).__name__)
print(type(elapsed_seconds).__name__)

The name “floating point” refers to a representation whose decimal point can
effectively move as the magnitude changes. It covers very small and very large
values with a finite amount of storage.

### Scientific notation keeps scale readable

In [ ]:
speed_of_light = 2.99792458e8
proton_width_m = 8.4e-16

print(speed_of_light)
print(proton_width_m)

`2.99792458e8` means \(2.99792458 \times 10^8\). `8.4e-16` means
\(8.4 \times 10^{-16}\). These are float literals, not strings.

### Integer and float arithmetic can mix

In [ ]:
distance_km = 15
hours = 2.5
average_speed = distance_km / hours

print(average_speed)
print(type(average_speed).__name__)

Python converts the integer to a compatible numeric representation for the
operation. The result is the float `6.0`. The original name `distance_km` still
refers to an integer; the conversion for the calculation does not reassign it.

### A float carries limited precision

In [ ]:
print(1.5 + 2.25)
print(9.0 / 4.0)
print(2.0 ** -3)

These results are `3.75`, `2.25`, and `0.125`. Fractions whose denominator is a
power of two can often be represented exactly in binary. Many familiar decimal
fractions cannot.

## 2. Why `0.1 + 0.2` is not exactly `0.3`

Run the classic comparison:

In [ ]:
result = 0.1 + 0.2

print(result)
print(result == 0.3)

Python commonly prints `0.30000000000000004`, and the equality comparison is
`False`. This is not random. A float stores a number in binary using a finite
number of bits. Just as decimal `1 / 3` becomes the repeating
`0.333333...`, decimal `0.1` repeats in binary.

The written decimal is converted to the nearest available binary float. Arithmetic
uses those stored approximations.

```{mermaid}
%%| echo: false
%%| eval: true
flowchart LR
  written["source text: 0.1"] --> convert["nearest binary float"]
  written2["source text: 0.2"] --> convert2["nearest binary float"]
  convert --> add["floating-point addition"]
  convert2 --> add
  add --> shown["shortest useful display<br/>0.30000000000000004"]
```

Python's normal display chooses the shortest decimal text that identifies the
stored float. It does not display every internal binary digit, but it sometimes
shows enough digits to reveal the approximation.

### Inspect more digits with formatting

In [ ]:
value = 0.1
print(f"{value:.20f}")

The output is close to `0.10000000000000000555`. The final digits make clear that
the stored value is near one tenth, not the exact decimal fraction one tenth.

Compare a binary-friendly value:

In [ ]:
value = 0.125
print(f"{value:.20f}")

One eighth is exactly representable in binary, so the displayed digits after
`125` are zeros.

### Approximation is usually appropriate for measurements

A ruler reading such as `12.4 cm` is already limited by the instrument and human
reading. A nearby binary representation is normally much more precise than the
measurement itself. The mistake is not using floats; the mistake is demanding
exact decimal behavior when the problem requires it.

### Checkpoint: float representation

## 3. Compare measurements with a stated tolerance

Exact equality asks whether two floats are the same stored number:

In [ ]:
calculated = 0.1 + 0.2
expected = 0.3

print(calculated == expected)
print(abs(calculated - expected))

The difference is extremely small but nonzero. For approximate quantities, ask
whether the difference is acceptable for the task.

Python's `math.isclose()` implements a clear tolerance comparison:

In [ ]:
import math

calculated = 0.1 + 0.2
expected = 0.3

print(math.isclose(calculated, expected))

The default tolerances consider these values close, so the output is `True`.
Default settings are convenient for demonstrations, but real tolerances should
come from a requirement such as instrument accuracy.

### Relative tolerance scales with magnitude

In [ ]:
import math

expected_distance = 1_000.0
measured_distance = 1_000.5

print(math.isclose(measured_distance, expected_distance, rel_tol=0.001))
print(math.isclose(measured_distance, expected_distance, rel_tol=0.0001))

A relative tolerance of `0.001` allows roughly one part in a thousand, which is
about `1.0` at this scale. A tolerance of `0.0001` allows roughly `0.1`, so the
half-unit difference is too large.

### Absolute tolerance matters near zero

Relative comparison alone is not useful when the expected value is zero:

In [ ]:
import math

sensor_offset = 0.0004

print(math.isclose(sensor_offset, 0.0))
print(math.isclose(sensor_offset, 0.0, abs_tol=0.001))

The second comparison states that offsets within `0.001` of zero are acceptable.
Choose the unit and tolerance together: `0.001` meters is not the same requirement
as `0.001` kilometers.

> **Tolerance is a domain rule**
Do not add an enormous tolerance merely to make a test pass. State what difference
the instrument, user, or calculation is allowed to have, then encode that rule.

### Exact equality is still useful

Float equality is appropriate when testing an exact sentinel that your own code
assigned, or when the contract truly requires the same stored value. The rule is
not “never use `==` with floats.” It is “do not use exact equality to express an
approximate requirement.”

In [ ]:
reading = 0.0
not_started = reading == 0.0
print(not_started)

If `reading` came from a physical sensor, an absolute tolerance might be the
better meaning. Context decides.

## 4. Rounding and formatting do different jobs

`round()` creates a numeric result:

In [ ]:
measurement = 12.34567
rounded_measurement = round(measurement, 2)

print(rounded_measurement)
print(type(rounded_measurement).__name__)
print(measurement)

The result is a float near `12.35`. The original value remains unchanged.

Formatting creates text for display:

In [ ]:
measurement = 12.34567
display_text = f"{measurement:.2f}"

print(display_text)
print(type(display_text).__name__)
print(measurement)

`display_text` is the string `"12.35"`. The stored measurement is still the
original float. Use formatting when the need is “show two decimal places.”

### Display precision does not change later arithmetic

In [ ]:
distance = 2.675
print(f"Displayed: {distance:.2f}")
print(distance * 10)

The formatted output does not feed back into `distance`. Later arithmetic uses the
stored float, not the characters shown to a person.

### `round()` follows the stored value and a tie rule

In [ ]:
print(round(2.5))
print(round(3.5))

Python normally uses “round half to even” for exact halfway cases: the results are
`2` and `4`. With values such as `2.675`, binary approximation means the stored
value may not lie on the decimal halfway point you imagine.

In [ ]:
print(round(2.675, 2))

Do not build legal or financial rules from an assumed school-rounding behavior.
The next lesson uses `Decimal` with an explicit rounding mode.

### Checkpoint: comparisons and display

## 5. Repeated operations can accumulate error

Adding one tenth ten times illustrates accumulation:

In [ ]:
total = 0.0
total = total + 0.1
total = total + 0.1
total = total + 0.1
total = total + 0.1
total = total + 0.1
total = total + 0.1
total = total + 0.1
total = total + 0.1
total = total + 0.1
total = total + 0.1

print(total)
print(total == 1.0)

This intentionally repeats statements because loops are taught later. The stored
result may be `0.9999999999999999`. Compare it according to the requirement:

In [ ]:
import math

print(math.isclose(total, 1.0))

For long sums, Python's `math.fsum()` can reduce accumulated rounding error. It
accepts a collection, so this example previews a list without requiring you to
modify it:

In [ ]:
import math

values = [0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1]
print(sum(values))
print(math.fsum(values))

Unit 3 will teach lists and iteration. For now, observe that algorithm choice can
affect accumulated error even when every input is the same.

## 6. Recognize infinity and NaN at numeric boundaries

Some calculations or external numeric systems produce special floats:

In [ ]:
positive_infinity = float("inf")
not_a_number = float("nan")

print(positive_infinity)
print(not_a_number)

Infinity is larger than ordinary finite floats:

In [ ]:
print(float("inf") > 1e308)

NaN means “not a number” and deliberately does not compare equal to itself:

In [ ]:
not_a_number = float("nan")
print(not_a_number == not_a_number)

The result is `False`. Do not use equality to detect NaN. The `math` module has
explicit checks:

In [ ]:
import math

print(math.isfinite(12.5))
print(math.isfinite(float("inf")))
print(math.isnan(float("nan")))

Ordinary division by zero in Python still raises `ZeroDivisionError`; it does not
automatically produce infinity:

In [ ]:
result = 1.0 / 0.0

Special values often arrive from scientific libraries or external data. Recognize
them now; later data and scientific courses will establish policies for them.

### Checkpoint: numeric boundaries

## 7. Build a sensor calibration report

An observatory expects a calibration weight of `10.0` grams. Three repeated
readings are `10.004`, `9.998`, and `10.002` grams. For this instrument, a mean
within `0.005` grams of the target is acceptable.

Use the supplied names without a loop:

In [ ]:
target_grams = 10.0
reading_1 = 10.004
reading_2 = 9.998
reading_3 = 10.002
tolerance_grams = 0.005

mean_grams = None
difference_grams = None
calibrated = None

Calculate the mean, absolute difference, and calibration fact. Then display the
mean and difference to three decimal places.

Use these assertions:

In [ ]:
import math

assert math.isclose(mean_grams, 10.001333333333333)
assert math.isclose(difference_grams, 0.0013333333333331865)
assert calibrated is True
assert f"{mean_grams:.3f}" == "10.001"
assert f"{difference_grams:.3f}" == "0.001"

> **Keep values and displays separate**
Calculate `calibrated` from the unformatted mean and the stated tolerance. Do not
compare display strings or round the measurements first.

After the ordinary case passes, change `reading_3` to `10.030`. Predict whether the
mean stays within tolerance, then recalculate from the top.

<details>
<summary>Hint: translate the requirement directly</summary>

The mean is the sum divided by `3`. The difference is `abs(mean_grams -
target_grams)`. `math.isclose()` can compare the two values with
`abs_tol=tolerance_grams` and `rel_tol=0.0` because the requirement is stated as an
absolute difference in grams.

</details>

<details class="solution">
<summary>Show one solution after testing your report</summary>

In [ ]:
import math

target_grams = 10.0
reading_1 = 10.004
reading_2 = 9.998
reading_3 = 10.002
tolerance_grams = 0.005

mean_grams = (reading_1 + reading_2 + reading_3) / 3
difference_grams = abs(mean_grams - target_grams)
calibrated = math.isclose(
    mean_grams,
    target_grams,
    rel_tol=0.0,
    abs_tol=tolerance_grams,
)

print(f"Mean: {mean_grams:.3f} g")
print(f"Difference: {difference_grams:.3f} g")
print(f"Calibrated: {calibrated}")

The multiline call is one expression inside open parentheses. The four-space
continuation indentation makes its arguments easy to inspect.

</details>

## 8. Check your precision decisions

Before moving on, answer these questions from the sensor artifact:

1. Why is float appropriate for measured grams but not automatically for exact
   decimal currency?
2. What tolerance did the requirement state, and in which unit?
3. Why should formatting happen after the calibration comparison?
4. How would the contract change if the tolerance were a percentage of the target?
5. Which explicit function detects a non-finite reading?

## Key points

> **Key points**
- Floats are finite binary approximations suitable for many measured quantities.
- Familiar decimal fractions such as `0.1` may not have exact finite binary forms.
- `math.isclose()` expresses approximate equality with relative and absolute
  tolerances.
- A tolerance belongs to the problem and must use the value's unit.
- `round()` produces a numeric result; formatting produces display text.
- Rounded output does not change the stored float used by later arithmetic.
- `math.isfinite()` and `math.isnan()` recognize special floating-point values.

## References

- [Python tutorial: floating-point arithmetic and its limitations](https://docs.python.org/3/tutorial/floatingpoint.html)
- [Python `math.isclose()` documentation](https://docs.python.org/3/library/math.html#math.isclose)
- [Python `math` checks for finite, infinite, and NaN values](https://docs.python.org/3/library/math.html#number-theoretic-and-representation-functions)
- [Python format-specification mini-language](https://docs.python.org/3/library/string.html#formatspec)